# Splitting a series into individual measurements

The `PupilSeries` can be split into a list of `PupilMeasurement` objects.

In [ ]:
import matplotlib.pyplot as plt

import piprkit

In [ ]:
ps = piprkit.load_real_series()

## Splitting the series

In [ ]:
print("Splitting series into individual measurements...")
pupil_measurements = ps.split(pre_stimulus_duration=10.0, post_stimulus_duration=30.0)
print(f"Created {len(pupil_measurements)} individual measurements")

for i, measurement in enumerate(pupil_measurements):
    time_range = measurement.get_time()[-1] - measurement.get_time()[0]
    stimulus_info = measurement.get_light_stimulus()
    baseline = measurement.metrics.baseline(duration=5.0)

    print(f"\nMeasurement {i + 1}:")
    print(f"  Time range: {time_range:.1f} seconds")
    print(f"  Data points: {len(measurement.get_time())}")
    print(f"  {baseline.get_name()}: {baseline:.3f}")
    print(f"  Light stimulus: {stimulus_info}")

## Baseline correction per measurement

Using a for loop the individual measurements can be processed. In this example we apply the baseline correction to each.

In [ ]:
fig, axes = plt.subplots(
    len(pupil_measurements),
    1,
    figsize=(8, 2.5 * len(pupil_measurements)),
    sharex=True,
    sharey=True,
)

if len(pupil_measurements) == 1:
    axes = [axes]  # Make it iterable for single subplot

colors = [
    "blue",
    "red",
    "green",
    "purple",
    "orange",
]

for i, measurement in enumerate(pupil_measurements):
    ax = axes[i]
    color = colors[i % len(colors)]

    measurement_corrected = measurement.filters.apply_baseline_correction(duration=5.0, inplace=False)
    baseline = measurement.metrics.baseline(duration=5.0)
    peak = measurement_corrected.metrics.peak_constriction()

    measurement_corrected.plot(
        ax=ax,
        label=f"Measurement {i + 1} ({baseline.get_label()}: {baseline:.2f})",
        color=color,
        alpha=0.8,
        linewidth=2,
    )

    measurement.plot_light_stimulus(
        ax=ax, alpha=0.4, color="yellow", label="Light Stimulus"
    )

    measurement_corrected.metrics.baseline(duration=5.0).plot(ax=ax, color="gray")
    peak.plot(ax=ax, color="black", label=f"{peak.get_label()}: {peak:.2f}")

    ax.set_title(f"Individual Measurement {i + 1} - Pupil Light Response")
    ax.set_ylabel("Relative Pupil Size")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

## All measurements on one time axis

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
for i, measurement in enumerate(pupil_measurements):
    start, end = measurement.get_light_stimulus().get_timing()
    print(f"Measurement {i + 1}: light stimulus from {start:.2f} s to {end:.2f} s")
    corrected = measurement.filters.apply_baseline_correction(duration=5.0, inplace=False)
    corrected.plot(ax=ax, label=f"Measurement {i + 1}", alpha=0.7)
pupil_measurements[0].plot_light_stimulus(ax=ax)
ax.set_title("All measurements, baseline corrected, 0 s at light offset")
ax.legend(fontsize=7)
plt.show()